# Proving inference runs on the GPU

[Back to the introduction](ch01_demo.ipynb). Run in order in a fresh kernel, in a browser with WebGPU (recent Chrome or Edge; Safari 26 and Firefox 141 on some platforms). This loads SmolLM2-135M-Instruct (q4, about 120 MB) twice: once with `backend="webgpu"`, once with `backend="wasm"`.

A successful `backend="webgpu"` load alone is weak evidence, so this notebook collects three independent proofs:

1. **A hardware GPU exists**: the browser exposes a WebGPU adapter that is not a software fallback.
2. **The model's tensors live in GPU memory**: with WebGPU, Transformers.js keeps the attention key/value cache in `gpu-buffer` tensors; with WASM every output is a `cpu` tensor. lmLite maps `backend="webgpu"` to the WebGPU execution provider only, so a load cannot silently fall back to WASM.
3. **It behaves like a GPU**: the same greedy generation, timed on both backends.

In [ ]:
import json
import time

import js
from lmlite import LLM

MODEL = dict(
    generator_model="HuggingFaceTB/SmolLM2-135M-Instruct",
    dtype="q4",  # the same weights on both backends, for a fair comparison
    max_new_tokens=32,
    do_sample=False,  # greedy, so both backends should produce the same text
    return_full_text=False,
)
PROMPT = "The three primary colors are"

## 1. Is there a hardware GPU?

`navigator.gpu.requestAdapter()` returns the GPU the browser would use. `fallback: true` means a software rasterizer (CPU emulation), which would make the rest of this notebook meaningless. Some browsers hide these fields: Firefox leaves the vendor and architecture empty to prevent fingerprinting and does not report `fallback` (shown as `None`). Proofs 2 and 3 do not depend on them.

In [ ]:
gpu_adapter = js.eval("""(async () => {
  const adapter = await navigator.gpu?.requestAdapter({powerPreference: 'high-performance'});
  if (!adapter) return JSON.stringify(null);
  const info = adapter.info ?? {};
  return JSON.stringify({vendor: info.vendor, architecture: info.architecture, description: info.description,
                         fallback: info.isFallbackAdapter ?? adapter.isFallbackAdapter ?? null});  // null: not reported
})""")
adapter = json.loads(str(await gpu_adapter()))
if adapter is None:
    raise RuntimeError("This browser exposes no WebGPU adapter; WebGPU inference is not possible here.")
print("WebGPU adapter:", adapter)
assert adapter["fallback"] is not True, "This is a software fallback adapter, not a hardware GPU"
if not adapter["vendor"] or adapter["fallback"] is None:
    print("This browser hides the adapter's identity or fallback status; rely on proofs 2 and 3.")

## 2. Where do the model's tensors live?

This probe runs one forward pass of the loaded model and counts its output tensors by [location](https://onnxruntime.ai/docs/tutorials/web/ep-webgpu.html#keep-tensor-data-on-gpu-io-binding): `gpu-buffer` tensors exist only in GPU memory, written by GPU kernels.

In [ ]:
tensor_locations = js.eval("""(async (llm) => {
  const {tokenizer, model} = llm.generator;
  const outputs = await model(tokenizer('Where do my tensors live?'));
  const locations = {};
  for (const tensor of Object.values(outputs)) {
    locations[tensor.location] = (locations[tensor.location] ?? 0) + 1;
    if (tensor.location === 'gpu-buffer') tensor.dispose();  // free the GPU memory now
  }
  return JSON.stringify(locations);
})""")

In [ ]:
gpu_llm = await LLM.create(backend="webgpu", **MODEL)
gpu_locations = json.loads(str(await tensor_locations(gpu_llm.js)))
print("WebGPU output tensors:", gpu_locations)
assert gpu_locations.get("gpu-buffer"), "No output stayed on the GPU: inference did not run on WebGPU"

## 3. Same generation, timed on both backends

Model loading and warm-up happen in `LLM.create()`, so only generation is timed. Small models on integrated GPUs may show a modest speed-up; proof 2 does not depend on it.

In [ ]:
start = time.perf_counter()
gpu_text = await gpu_llm.generate(PROMPT)
gpu_seconds = time.perf_counter() - start
await gpu_llm.dispose()
print(f"WebGPU: {gpu_seconds:.2f} s\n{PROMPT}{gpu_text}")

In [ ]:
wasm_llm = await LLM.create(backend="wasm", **MODEL)
print("WASM output tensors:", json.loads(str(await tensor_locations(wasm_llm.js))))
start = time.perf_counter()
wasm_text = await wasm_llm.generate(PROMPT)
wasm_seconds = time.perf_counter() - start
await wasm_llm.dispose()
print(f"WASM: {wasm_seconds:.2f} s\n{PROMPT}{wasm_text}")

In [ ]:
print(f"WebGPU {gpu_seconds:.2f} s vs WASM {wasm_seconds:.2f} s: {wasm_seconds / gpu_seconds:.1f}x")
print("Same greedy text on both backends:", gpu_text == wasm_text)
print(f"Verdict: inference ran on the {(adapter['vendor'] + ' ') if adapter['vendor'] else ''}GPU ({gpu_locations['gpu-buffer']} tensors in GPU memory).")

On an Apple M-series Mac (Chrome, Metal), this printed 60 `gpu-buffer` tensors for WebGPU versus only `cpu` tensors for WASM, and 0.24 s versus 4.1 s for 32 tokens, with identical text. q4 kernels differ between backends, so on other GPUs the texts can diverge after a few tokens without anything being wrong.